To Do:

- x what attributes are relevant for the user in total?
- x which can bestored on the Dataset Level?
- x which can be stored on the DataArray Level?
- x make implementation
- o write tests for the storrage process

## User Relevant Attributes

- what noise is underlying?
- what trend is underlying?
- how many shifts are in the PONDS
- for each shift:
    - what kind is it? -> to select matching detector
    - which parameters were used to generate it? -> reproducability
    - if not single:
        - which blobgenerator was used?
        - which parameters were used?   -> reproducability

## Distributing onto Levels

| Dataset Level         | DataArray Level     |
|-----------------------|---------------------|
| - noise               | - type of shift     |
| - trend               | - shift parameters  |
| - number of shifts    | - type of blob      |
| - type of shift       | - blob parameters   |

## Try out Attribute Addition

In [1]:
import numpy as np
import xarray as xr

# Create sample data
dims = ('time', 'x', 'y')
shape = (10, 3, 2)
data1 = np.random.rand(*shape)
data2 = np.random.choice([True, False], size=shape)
data3 = np.random.choice([True, False], size=shape)

# Create DataArrays
da1 = xr.DataArray(data1, dims=dims, attrs={
    'noise': 0.01,
    'background': 0.005,
})
da2 = xr.DataArray(data2, dims=dims, attrs={
    'shift_type': 'SIGMOID',
    'shift_params': {
        "shift_steepness": 1,
        "shift_magnitude": 1,
        "sigma_time": 0,
    },
    'blob_type': 'GAUSS',
    'blob_params': {
        "sigma_x": 0.1,
        "sigma_y": 0.1,
    }
})
da3 = xr.DataArray(data3, dims=dims, attrs={
    'shift_type': 'LORENZ',
    'shift_params': {
        "a": 1,
        "b": 1,
    },
    'blob_type': 'GAUSS',
    'blob_params': {
        "sigma_x": 0.1,
        "sigma_y": 0.1,
    }
})

# Combine into a Dataset
ds = xr.Dataset({'ts': da1, 'c1': da2, 'c2': da3}, attrs={
    'noise': 0.01,
    'background': 0.005,
    'num_clusters': 2,
    'c1': 'SIGMOID',
    'c2': 'LORENZ'
})

ds

<xarray.Dataset> Size: 600B
Dimensions:  (time: 10, x: 3, y: 2)
Dimensions without coordinates: time, x, y
Data variables:
    ts       (time, x, y) float64 480B 0.4118 0.8461 0.9366 ... 0.5887 0.7674
    c1       (time, x, y) bool 60B False False True False ... False True False
    c2       (time, x, y) bool 60B False True False True ... False False False
Attributes:
    noise:         0.01
    background:    0.005
    num_clusters:  2
    c1:            SIGMOID
    c2:            LORENZ

In [2]:
ts = ds['ts']
c1 = ds['c1']
c2 = ds['c2']

ts

<xarray.DataArray 'ts' (time: 10, x: 3, y: 2)> Size: 480B
array([[[0.41178022, 0.84607148],
        [0.93658565, 0.75660001],
        [0.23670224, 0.55785463]],

       [[0.18674428, 0.33440729],
        [0.76177544, 0.20533429],
        [0.45302093, 0.58328708]],

       [[0.02181341, 0.41437538],
        [0.34161234, 0.00816774],
        [0.86585026, 0.47707192]],

       [[0.59335537, 0.20496295],
        [0.09750218, 0.18555914],
        [0.10325436, 0.9640937 ]],

       [[0.5634053 , 0.24769371],
        [0.80374015, 0.29935492],
        [0.06056995, 0.4751858 ]],

       [[0.44443743, 0.01153078],
        [0.53132949, 0.58075085],
        [0.5537771 , 0.71001176]],

       [[0.17000101, 0.21906568],
        [0.00971646, 0.03095551],
        [0.08649508, 0.37536629]],

       [[0.99997668, 0.77303314],
        [0.83915869, 0.1475059 ],
        [0.77747069, 0.5302346 ]],

       [[0.90191354, 0.96609898],
        [0.23062799, 0.64975867],
        [0.22984329, 0.9318432 ]],

       [[0.77338755, 0.18479463],
        [0.27366505, 0.10097771],
        [0.58872811, 0.76738798]]])
Dimensions without coordinates: time, x, y
Attributes:
    noise:       0.01
    background:  0.005

## Try Out in PONDS

In [3]:
from ponds import PONDS, GAUSS, SIGMOID
import matplotlib.pyplot as plt

In [4]:
background_noise = 0.05
background_trend = 0.002

pd = PONDS(
    time_size=200,
    background_noise=background_noise,
    background_trend=background_trend,
)


# adding a random shift to the time series
pd.add_shift(
    random_seed=2,
    shift_method=SIGMOID(),
)


# adding a random shift to the time series
pd.add_shift(
    random_seed=2,
    shift_method=SIGMOID(),
)

pd.data

<xarray.Dataset> Size: 4kB
Dimensions:  (time: 200, lat: 1, lon: 1)
Coordinates:
  * time     (time) float64 2kB 0.0 1.005 2.01 3.015 ... 197.0 198.0 199.0 200.0
  * lat      (lat) float64 8B -90.0
  * lon      (lon) float64 8B -180.0
Data variables:
    ts       (time, lat, lon) float64 2kB 0.08122 -0.02858 ... 2.483 2.496
    c1       (time, lat, lon) bool 200B False False False ... False False False
    c2       (time, lat, lon) bool 200B False False False ... False False False
Attributes:
    random_seed:   1
    trend:         0.002
    noise:         0.05
    num_clusters:  2
    c1:            SIGMOID
    c2:            SIGMOID

In [5]:
ts = pd.data['ts']
c1 = pd.data['c1']
c2 = pd.data['c2']

c1

<xarray.DataArray 'c1' (time: 200, lat: 1, lon: 1)> Size: 200B
array([[[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

...

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]]])
Coordinates:
  * time     (time) float64 2kB 0.0 1.005 2.01 3.015 ... 197.0 198.0 199.0 200.0
  * lat      (lat) float64 8B -90.0
  * lon      (lon) float64 8B -180.0
Attributes:
    shift_type:    SIGMOID
    shift_params:  {'random_seed': 1, 'shift_steepness': 2.125533007053861, '...
    shape_type:    SINGLE
    shape_params:  {}

In [6]:
# initialize PONDS object
pd = PONDS(
    lat_size=20,
    lon_size=40,
    time_size=200,
    background_noise=0.05,
    background_trend=0.0002,
)

# add random shift
pd.add_shift(
    random_seed=1,
    shape_method=GAUSS(random_seed=1),
    shift_method=SIGMOID(),
)

# add shift with custom parameters
shift_dict = {
    "shift_steepness": 0.2,
    "shift_magnitude": 0.5,
}
pd.add_shift(
    random_seed=4,
    shape_method=GAUSS(random_seed=4),
    shift_method=SIGMOID(**shift_dict),
)


pd.data

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 200, lat: 20, lon: 40)
Coordinates:
  * time     (time) float64 2kB 0.0 1.005 2.01 3.015 ... 197.0 198.0 199.0 200.0
  * lat      (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon      (lon) float64 320B -180.0 -170.8 -161.5 ... 161.5 170.8 180.0
Data variables:
    ts       (time, lat, lon) float64 1MB 0.08122 -0.02004 ... 0.04573 0.02238
    c1       (time, lat, lon) bool 160kB False False False ... False False False
    c2       (time, lat, lon) bool 160kB False False False ... False False False
Attributes:
    random_seed:   1
    trend:         0.0002
    noise:         0.05
    num_clusters:  2
    c1:            SIGMOID
    c2:            SIGMOID

In [7]:
ts = pd.data['ts']
c1 = pd.data['c1']
c2 = pd.data['c2']

c1

<xarray.DataArray 'c1' (time: 200, lat: 20, lon: 40)> Size: 160kB
array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
...
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]]])
Coordinates:
  * time     (time) float64 2kB 0.0 1.005 2.01 3.015 ... 197.0 198.0 199.0 200.0
  * lat      (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon      (lon) float64 320B -180.0 -170.8 -161.5 ... 161.5 170.8 180.0
Attributes:
    shift_type:    SIGMOID
    shift_params:  {'random_seed': 1, 'shift_steepness': 2.125533007053861, '...
    shape_type:    GAUSS
    shape_params:  {'random_seed': 1, 'center_lat': -13.276479247588156, 'cen...

## Reproducibility


In [4]:
import numpy as np
from ponds import PONDS, GAUSS, SIGMOID

# 1) setup variables
# - basic parameters
lat_size = 30
lon_size = 40
time_size = 30
noise = 0.01
trend = 0.005
random_seed = 4
# - create PONDS object
ponds = PONDS(lat_size=lat_size,
                lon_size=lon_size,
                time_size=time_size,
                background_noise=noise,
                background_trend=trend,
                random_seed=random_seed)
# - add a shift using SIGMOID and GAUSS
ponds.add_shift(
    shape_method=GAUSS(random_seed=6),
    shift_method=SIGMOID(random_seed=4),)

# 2) call function
# - extract all parameters from the Dataset
ponds_dict_ex = ponds.data.attrs["ponds_params"]
shape_dict_ex = ponds.data.c1.attrs["shape_params"]
shift_dict_ex = ponds.data.c1.attrs["shift_params"]
# - reproduce the shift using the stored parameters
ponds_rep = PONDS(**ponds_dict_ex)
ponds_rep.add_shift(
    shape_method=GAUSS(**shape_dict_ex),
    shift_method=SIGMOID(**shift_dict_ex))


noise: 0.9340596780273533
noise: 0.09446449835144466
noise: 0.9453687199297687
noise: 0.4296319873487293
noise: 0.3954576491945416
noise: -0.5678210088392472
noise: 0.9525489095524835
noise: -0.9875394895908203
noise: -0.4940352752331121
noise: -0.13041693519110842
noise: 0.5587658435875049
noise: -0.6046298507994938
noise: 0.7259864711984445
noise: 0.9668013543506255
noise: -0.6723155171906026
noise: 0.19466788786571843
noise: -0.98202780466489
noise: -0.22685743471274122
noise: -0.9116798841370009
noise: 0.9133059354284718
noise: -0.1277067062404047
noise: 0.9340596780273533
noise: 0.09446449835144466
noise: 0.9453687199297687
noise: 0.4296319873487293
noise: 0.3954576491945416
noise: -0.5678210088392472
noise: 0.9525489095524835
noise: -0.9875394895908203
noise: -0.4940352752331121
noise: -0.13041693519110842
noise: 0.5587658435875049
noise: -0.6046298507994938
noise: 0.7259864711984445
noise: 0.9668013543506255
noise: -0.6723155171906026
noise: 0.19466788786571843
noise: -0.9820278

In [5]:

# 3) check results
# - check if attributes are equal
print("Checking if attributes are equal...")
print(">> Original attributes:\n", ponds.data.attrs)
print(">> Reproduced attributes:\n", ponds_rep.data.attrs)
print("")
print(">> Original shape parameters:\n", ponds.data.c1.attrs["shape_params"])
print(">> Reproduced shape parameters:\n", ponds_rep.data.c1.attrs["shape_params"])
print("")
print(">> Original shift parameters:\n", ponds.data.c1.attrs["shift_params"])
print(">> Reproduced shift parameters:\n", ponds_rep.data.c1.attrs["shift_params"])
print("")

# - check if the time series are equal
print("Checking if time series are equal...")
#print(np.mean(ponds.data.ts.values - ponds_rep.data.ts.values) <= 1e-3)
print(np.array_equal(ponds.data.ts.values, ponds_rep.data.ts.values))
#print(">> Original time series:\n", ponds.data.ts.values)
#print(">> Reproduced time series:\n", ponds_rep.data.ts.values)

counter = 0
delta = []
for i in range(lat_size):
    for j in range(lon_size):
        for t in range(time_size):
            if ponds.data.ts.values[t, i, j] != ponds_rep.data.ts.values[t, i, j]:
                counter += 1
                delta.append(np.abs(ponds.data.ts.values[t, i, j] - ponds_rep.data.ts.values[t, i, j]))
                #print(f"Difference found at ({i}, {j}, {t}): {ponds.data.ts.values[t, i, j]} != {ponds_rep.data.ts.values[t, i, j]}")
print(f"Differend values found: {counter}/{lat_size * lon_size * time_size}  ({counter / (lat_size * lon_size * time_size) * 100:.2f}%)")
print(f"Mean difference of mismatches: {np.mean(delta):.4f}")

Checking if attributes are equal...
>> Original attributes:
 {'ponds_params': {'lat_size': 30, 'lon_size': 40, 'time_size': 30, 'background_trend': 0.005, 'background_noise': 0.01, 'random_seed': 4, 'cluster_key': 'c'}, 'num_clusters': 1, 'c1': 'SIGMOID'}
>> Reproduced attributes:
 {'ponds_params': {'lat_size': 30, 'lon_size': 40, 'time_size': 30, 'background_trend': 0.005, 'background_noise': 0.01, 'random_seed': 4, 'cluster_key': 'c'}, 'num_clusters': 1, 'c1': 'SIGMOID'}

>> Original shape parameters:
 {'center_lat': 62.85762422976026, 'center_lon': -60.48727009157621, 'sigma_lat': 18.212291230578316, 'sigma_lon': 16.042415643131246}
>> Reproduced shape parameters:
 {'center_lat': 62.85762422976026, 'center_lon': -60.48727009157621, 'sigma_lat': 18.212291230578316, 'sigma_lon': 16.042415643131246}

>> Original shift parameters:
 {'central_time': 28, 'sigma_time': 1.0, 'shift_steepness': 2.950544758520515, 'shift_magnitude': 1.0047232249175722}
>> Reproduced shift parameters:
 {'centr

After playing around with the setup above, it seems like the error is only related to the usage SIGMOID method.

### Reproducibility of SIGMOID

In [6]:
R = 1
shift_dict = {#'random_seed': 2,
              'shift_steepness': 1.6144624340609357,
              'shift_magnitude': 1.0279918792240115,
              'sigma_time': 1.0}

# 1) setup variables
time = np.arange(0, 30, 1)  # Example time array
shift_time = 5

for _ in range(R):
    #random_seed = np.random.randint(0, 1000)

    # 2) call function
    sigmoid1 = SIGMOID(**shift_dict)  # <- testing this
    shift_effect1, st1 = sigmoid1.make_shift(
        time,
    )  # <- testing this: no shift parameters provided, so random values are used

    sigmoid2 = SIGMOID(**shift_dict)  # <- testing this
    shift_effect2, st2 = sigmoid2.make_shift(
        time,
    )  # <- testing this: no shift parameters provided, so random values are used

    # 3) check results
    #assert np.array_equal(shift_effect1, shift_effect2), (
    #    "Shifts with same seed should produce identical results."
    #)

    #print(f"Random seed: {random_seed}, Shift effect: {shift_effect1}, {shift_effect2}")
    print(f"Are the shift effects equal?\n>> {np.array_equal(shift_effect1, shift_effect2)}")
    print(f"\nShift Dictionaries:\n>> {shift_dict}\n>> {sigmoid1.param_dict}\n>> {sigmoid2.param_dict}")
    print(f"\nShift times:\n>> S1: {st1}\n>> S2: {st2}")

noise: 0.4406489868843162
noise: 0.4406489868843162
Are the shift effects equal?
>> True

Shift Dictionaries:
>> {'shift_steepness': 1.6144624340609357, 'shift_magnitude': 1.0279918792240115, 'sigma_time': 1.0}
>> {'type': 'SIGMOID', 'central_time': 12, 'sigma_time': 1.0, 'shift_steepness': 1.6144624340609357, 'shift_magnitude': 1.0279918792240115}
>> {'type': 'SIGMOID', 'central_time': 12, 'sigma_time': 1.0, 'shift_steepness': 1.6144624340609357, 'shift_magnitude': 1.0279918792240115}

Shift times:
>> S1: 12
>> S2: 12


### Test Variability of SIGMOID

In [7]:
import numpy as np
from ponds import SIGMOID

# 1) setup variables
err_tol = 1e-1  # tolerance for numerical errors
lat_size = 100
lon_size = 100
time = np.arange(0, 20, 1)  # Example time array
shift_time = 5

custom_sigma_time = 3.0
custom_steepness = 5.0
custom_magnitude = 2.0

# 2) call function
t_shifts = np.zeros((lat_size, lon_size))
sigmoid_shift = SIGMOID(
    central_time=shift_time,
    sigma_time=custom_sigma_time,
    shift_steepness=custom_steepness,
    shift_magnitude=custom_magnitude,
)
for i in range(lat_size):
    for j in range(lon_size):
        _, t_shifts[i, j] = sigmoid_shift.make_shift(time)

# 3) check results
# - extract the shift parameters from the generated data
# -- sigma_time
print(t_shifts)
t0_std = np.std(t_shifts.flatten())
t0_std_expected = (
    custom_sigma_time * 2 / np.sqrt(12)
)  # for a uniform distribution, the standard deviation is (b-a)/sqrt(12), where a and b are the bounds of the distribution
assert np.isclose(t0_std, t0_std_expected, atol=err_tol), (
    f"Expected standard deviation {t0_std_expected} not used, got {t0_std}."
)
# assert np.abs(t0_std - custom_sigma_time) > 0, (
#    f"Custom sigma_time {custom_sigma_time} not used, got {t0_std}."
# )

# -- shift_steepness

# assert np.allclose(params["sigma_time"], custom_sigma_time), "Custom sigma_time not used"
# assert np.allclose(params["shift_steepness"], custom_steepness), "Custom shift_steepness not used"
# assert np.allclose(params["shift_magnitude"], custom_magnitude), "Custom shift_magnitude not used"


noise: -0.4978679717845562
noise: 1.3219469606529488
noise: -2.9993137510959307
noise: -1.1860045642089614
noise: -2.119464655097322
noise: -2.445968431387213
noise: -1.8824387317339746
noise: -0.9266356377417138
noise: -0.6193951546159804
noise: 0.23290040402014167
noise: -0.48483291358023095
noise: 1.1113170023805568
noise: -1.7732865016108954
noise: 2.268704618345673
noise: -2.835674440812443
noise: 1.0228050610704136
noise: -0.4961711857972384
noise: 0.3521389706745097
noise: -2.1576783684285976
noise: -1.8113910654907273
noise: 1.8044674120532198
noise: 2.809569454316385
noise: -1.119454931044543
noise: 1.1539356940158845
noise: 2.2583349137762294
noise: 2.3676399810230837
noise: -2.4897347317813328
noise: -2.7656713006027056
noise: -1.9810174826125866
noise: 2.2688550205764786
noise: -2.409918997001699
noise: -0.4733542499696872
noise: 2.7473371809030116
noise: 0.19899170983810244
noise: 1.1512626837028401
noise: -1.1069062139636223
noise: 1.119005566089502
noise: 2.0077540313842